In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv
/kaggle/input/datasets/utkarshx27/heart-disease-diagnosis-dataset/dataset_heart.csv


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import QuantileTransformer

def target_encode(train, test, target, cat_cols, n_splits=5, alpha=20):
    train = train.copy()
    test = test.copy()
    global_mean = train[target].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    for col in cat_cols:
        oof = np.zeros(len(train))
        test_enc = np.zeros(len(test))

        for train_idx, val_idx in kf.split(train):
            tr, val = train.iloc[train_idx], train.iloc[val_idx]

            stats = tr.groupby(col)[target].agg(['mean', 'count'])

            smooth = (
                    (stats['mean'] * stats['count'] + global_mean * alpha) /
                    (stats['count'] + alpha)
            )

            oof[val_idx] = val[col].map(smooth).fillna(global_mean)
            test_enc += test[col].map(smooth).fillna(global_mean) / n_splits

        train[col + "_te"] = oof
        test[col + "_te"] = test_enc

    return train, test


train_data = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test_data = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")
zov = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")

train_data = train_data.drop("id", axis=1)
test_data = test_data.drop("id", axis=1)

Y = (train_data["Heart Disease"] == "Presence").astype(int)
train_data = train_data.drop("Heart Disease", axis=1)
train_data["target"] = Y

num_feat = ["Age", "BP", "Cholesterol", "Max HR", "ST depression", "Number of vessels fluro"]
cat_feat = ["Sex", "Chest pain type", "FBS over 120", "EKG results", "Exercise angina", "Slope of ST", "Thallium"]

for col in num_feat:
    train_data[col] = train_data[col].fillna(train_data[col].median())
    test_data[col] = test_data[col].fillna(train_data[col].median())

for col in cat_feat:
    train_data[col] = train_data[col].fillna("None")
    test_data[col] = test_data[col].fillna("None")

X, X_test = target_encode(
    train_data,
    test_data,
    target="target",
    cat_cols=cat_feat
)
def preprocess(df):
    df["RPP"] = df["Max HR"] * df["BP"] 
    df["Age_MaxHR_Diff"] = (220 - df["Age"]) - df["Max HR"]
    df["ST_per_BP"] = df["ST depression"] / (df["BP"] + 1)
    df["Chol_Age_Ratio"] = df["Cholesterol"] / (df["Age"] + 1)
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    cat_cols = df.select_dtypes(include=['object']).columns
    df[cat_cols] = df[cat_cols].fillna("None")
    
    return df

qt = QuantileTransformer(output_distribution='normal', random_state=42)
X[num_feat] = qt.fit_transform(X[num_feat])
X_test[num_feat] = qt.transform(X_test[num_feat])
X["Age_BP"] = X["Age"] * X["BP"]
X["HR_Age"] = X["Max HR"] / X["Age"]
X["ST_BP"] = X["ST depression"] * X["BP"]
X["Chol_HR"] = X["Cholesterol"] / X["Max HR"]
X["ChestPain_Age"] = X["Chest pain type_te"] * X["Age"]
X["Thal_ST"] = X["Thallium_te"] * X["ST depression"]

X_test["Age_BP"] = X_test["Age"] * X_test["BP"]
X_test["HR_Age"] = X_test["Max HR"] / X_test["Age"]
X_test["ST_BP"] = X_test["ST depression"] * X_test["BP"]
X_test["Chol_HR"] = X_test["Cholesterol"] / X_test["Max HR"]
X_test["ChestPain_Age"] = X_test["Chest pain type_te"] * X_test["Age"]
X_test["Thal_ST"] = X_test["Thallium_te"] * X_test["ST depression"]
X=preprocess(X)
X_test=preprocess(X_test)
X = X.drop("target", axis=1)

X[num_feat] = X[num_feat].astype(np.float32)
X_test[num_feat] = X_test[num_feat].astype(np.float32)

X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)



In [3]:
import optuna
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
import numpy as np

def objective(trial):
    params = {
        "iterations": 2400,
        "learning_rate": trial.suggest_float("learning_rate", 0.008, 0.022, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 40.0),
        "bootstrap_type": "Bernoulli",
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "random_strength": trial.suggest_float("random_strength", 4.0, 10.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 40),
        "task_type": "GPU",
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "early_stopping_rounds": 400,
        "verbose": False
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = []

    for train_idx, val_idx in skf.split(X, Y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = Y.iloc[train_idx], Y.iloc[val_idx]

        model = CatBoostClassifier(**params)
        model.fit(X_train, y_train, eval_set=(X_val, y_val), cat_features=cat_feat)
        best_auc = model.get_best_score()["validation"]["AUC"]
        cv_scores.append(best_auc)

    return np.mean(cv_scores)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20) 

print("Best params:", study.best_params)
print("Best AUC:", study.best_value)

[I 2026-02-27 14:36:50,664] A new study created in memory with name: no-name-8c68aa65-02e7-4e97-ad2b-9c2765b37bf2
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-02-27 14:41:00,791] Trial 0 finished with value: 0.9550908406575521 and parameters: {'learning_rate': 0.018872054454554282, 'depth': 7, 'l2_leaf_reg': 9.546743551263509, 'subsample': 0.7647842519164216, 'random_strength': 8.542053962086799, 'min_data_in_leaf': 24}. Best is trial 0 with value: 0.9550908406575521.
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-02-27 14:45:15,900] Trial 1 finished with value: 0.9550689458847046 and parameters: {'learning_rate': 0.012503446751453918, 'depth': 7, 

Best params: {'learning_rate': 0.02002386811936143, 'depth': 6, 'l2_leaf_reg': 23.724075870603752, 'subsample': 0.8942755759011857, 'random_strength': 7.910889239339775, 'min_data_in_leaf': 16}
Best AUC: 0.9551824927330017


In [4]:
params = {
    "loss_function":"Logloss",
    "eval_metric": "AUC",
    "iterations": 12000,
    "learning_rate": 0.0112,
    "depth": 9,
    "l2_leaf_reg": 28,
    "bootstrap_type": "Bernoulli",
    "random_strength": 4.60,
    "border_count": 254,
    "od_type": "Iter",
    "od_wait": 200,
    "random_seed": 42,
    "subsample": 0.8,
    "task_type" : "GPU",
    "one_hot_max_size": 17,
    "min_data_in_leaf":20,
    "devices" : "0",
    "grow_policy": "Lossguide",
    "verbose": 200,
    "auto_class_weights": "Balanced"
}

n_splits = 10
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

test_preds = np.zeros(len(X_test))
oof_preds = np.zeros(len(X))
fold_test_preds = np.zeros((n_splits, len(X_test)))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, Y)):
    print(f"\n===== Fold {fold + 1} =====")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = Y.iloc[train_idx], Y.iloc[val_idx]

    model = CatBoostClassifier(**params)

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=cat_feat,
        early_stopping_rounds=1500,
        use_best_model=True,
        verbose=200
    )

    oof_preds[val_idx] = (model.predict_proba(X_val)[:, 1] )
    test_preds += model.predict_proba(X_test)[:, 1] / n_splits
    fold_test_preds[fold] = model.predict_proba(X_test)[:, 1]


oof_auc = roc_auc_score(Y, oof_preds)
print("\nOOF AUC:", oof_auc)



===== Fold 1 =====


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9314896	best: 0.9314896 (0)	total: 107ms	remaining: 21m 19s
200:	test: 0.9524497	best: 0.9524497 (200)	total: 3.26s	remaining: 3m 11s
400:	test: 0.9538082	best: 0.9538082 (400)	total: 6.25s	remaining: 3m
600:	test: 0.9542140	best: 0.9542140 (600)	total: 9.14s	remaining: 2m 53s
800:	test: 0.9545596	best: 0.9545600 (799)	total: 12s	remaining: 2m 47s
1000:	test: 0.9547889	best: 0.9547890 (999)	total: 14.8s	remaining: 2m 42s
1200:	test: 0.9549061	best: 0.9549061 (1200)	total: 17.5s	remaining: 2m 37s
1400:	test: 0.9549693	best: 0.9549693 (1400)	total: 20.2s	remaining: 2m 32s
1600:	test: 0.9550176	best: 0.9550176 (1600)	total: 22.8s	remaining: 2m 28s
1800:	test: 0.9550436	best: 0.9550438 (1799)	total: 25.4s	remaining: 2m 23s
2000:	test: 0.9550598	best: 0.9550602 (1958)	total: 28s	remaining: 2m 19s
2200:	test: 0.9550652	best: 0.9550652 (2200)	total: 30.6s	remaining: 2m 16s
2400:	test: 0.9550720	best: 0.9550759 (2331)	total: 33.1s	remaining: 2m 12s
2600:	test: 0.9550752	best: 0.9550

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9323513	best: 0.9323513 (0)	total: 25.2ms	remaining: 5m 2s
200:	test: 0.9535618	best: 0.9535618 (200)	total: 3.17s	remaining: 3m 6s
400:	test: 0.9549088	best: 0.9549088 (400)	total: 6.26s	remaining: 3m 1s
600:	test: 0.9553850	best: 0.9553850 (600)	total: 9.17s	remaining: 2m 53s
800:	test: 0.9556843	best: 0.9556843 (800)	total: 12s	remaining: 2m 47s
1000:	test: 0.9558602	best: 0.9558602 (1000)	total: 14.8s	remaining: 2m 42s
1200:	test: 0.9559737	best: 0.9559740 (1198)	total: 17.5s	remaining: 2m 37s
1400:	test: 0.9560314	best: 0.9560320 (1389)	total: 20.1s	remaining: 2m 32s
1600:	test: 0.9560634	best: 0.9560637 (1599)	total: 22.8s	remaining: 2m 27s
1800:	test: 0.9560808	best: 0.9560810 (1767)	total: 25.4s	remaining: 2m 24s
2000:	test: 0.9560984	best: 0.9560990 (1974)	total: 28s	remaining: 2m 20s
2200:	test: 0.9561050	best: 0.9561066 (2183)	total: 30.6s	remaining: 2m 16s
2400:	test: 0.9561033	best: 0.9561077 (2260)	total: 33.2s	remaining: 2m 12s
2600:	test: 0.9561044	best: 0.95

...

0:	test: 0.9320337	best: 0.9320337 (0)	total: 25.8ms	remaining: 5m 9s
200:	test: 0.9530724	best: 0.9530724 (200)	total: 3.17s	remaining: 3m 6s
400:	test: 0.9545784	best: 0.9545784 (400)	total: 6.13s	remaining: 2m 57s
600:	test: 0.9550639	best: 0.9550639 (600)	total: 8.99s	remaining: 2m 50s
800:	test: 0.9554027	best: 0.9554027 (800)	total: 11.8s	remaining: 2m 45s
1000:	test: 0.9556352	best: 0.9556352 (1000)	total: 14.6s	remaining: 2m 39s
1200:	test: 0.9557609	best: 0.9557609 (1200)	total: 17.3s	remaining: 2m 35s
1400:	test: 0.9558331	best: 0.9558336 (1393)	total: 19.9s	remaining: 2m 30s
1600:	test: 0.9558796	best: 0.9558799 (1592)	total: 22.6s	remaining: 2m 26s
1800:	test: 0.9558976	best: 0.9558976 (1800)	total: 25.2s	remaining: 2m 22s
2000:	test: 0.9559145	best: 0.9559152 (1998)	total: 27.8s	remaining: 2m 18s
2200:	test: 0.9559222	best: 0.9559228 (2197)	total: 30.4s	remaining: 2m 15s
2400:	test: 0.9559239	best: 0.9559259 (2360)	total: 33s	remaining: 2m 11s
2600:	test: 0.9559277	best: 0

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9322247	best: 0.9322247 (0)	total: 25.5ms	remaining: 5m 5s
200:	test: 0.9536331	best: 0.9536331 (200)	total: 3.1s	remaining: 3m 2s
400:	test: 0.9550300	best: 0.9550300 (400)	total: 6.11s	remaining: 2m 56s
600:	test: 0.9554451	best: 0.9554451 (600)	total: 9.05s	remaining: 2m 51s
800:	test: 0.9557455	best: 0.9557455 (799)	total: 11.9s	remaining: 2m 46s
1000:	test: 0.9559802	best: 0.9559802 (1000)	total: 14.7s	remaining: 2m 41s
1200:	test: 0.9560880	best: 0.9560882 (1199)	total: 17.5s	remaining: 2m 37s
1400:	test: 0.9561331	best: 0.9561332 (1398)	total: 20.2s	remaining: 2m 32s
1600:	test: 0.9561639	best: 0.9561648 (1590)	total: 22.7s	remaining: 2m 27s
1800:	test: 0.9561816	best: 0.9561825 (1794)	total: 25.3s	remaining: 2m 23s
2000:	test: 0.9561862	best: 0.9561889 (1926)	total: 27.9s	remaining: 2m 19s
2200:	test: 0.9561912	best: 0.9561948 (2111)	total: 30.5s	remaining: 2m 15s
2400:	test: 0.9561851	best: 0.9561948 (2111)	total: 33s	remaining: 2m 12s
2600:	test: 0.9561797	best: 0.

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9300803	best: 0.9300803 (0)	total: 25.5ms	remaining: 5m 6s
200:	test: 0.9523204	best: 0.9523204 (200)	total: 3.15s	remaining: 3m 4s
400:	test: 0.9538604	best: 0.9538604 (400)	total: 6.14s	remaining: 2m 57s
600:	test: 0.9543839	best: 0.9543839 (600)	total: 9.05s	remaining: 2m 51s
800:	test: 0.9547256	best: 0.9547256 (800)	total: 11.9s	remaining: 2m 46s
1000:	test: 0.9549350	best: 0.9549350 (1000)	total: 14.7s	remaining: 2m 41s
1200:	test: 0.9550624	best: 0.9550627 (1199)	total: 17.4s	remaining: 2m 36s
1400:	test: 0.9551264	best: 0.9551264 (1400)	total: 20.1s	remaining: 2m 31s
1600:	test: 0.9551705	best: 0.9551706 (1599)	total: 22.8s	remaining: 2m 27s
1800:	test: 0.9552104	best: 0.9552104 (1800)	total: 25.4s	remaining: 2m 23s
2000:	test: 0.9552163	best: 0.9552196 (1968)	total: 28s	remaining: 2m 20s
2200:	test: 0.9552219	best: 0.9552245 (2152)	total: 30.7s	remaining: 2m 16s
2400:	test: 0.9552201	best: 0.9552245 (2152)	total: 33.3s	remaining: 2m 13s
2600:	test: 0.9552180	best: 0

In [5]:

final_pred=test_preds

submission = pd.DataFrame(
    np.column_stack((zov["id"].astype(int), final_pred)),
    columns=["id", "Heart Disease"]
)
submission.to_csv("submission1389.csv", index=False)